# VocEd Lab 3 — Grid Search, Bayesian Optimisation, and a Processing Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab3_grid_search_bayesian_optimisation_pipeline.ipynb)

In Lab 2 we built a two-threshold segmenter, tuned **one** threshold (`t_nucleus`) on training
images, and kept `t_background = 0.85` fixed.  This lab asks three questions:

- *What if we tune **both** thresholds?*  We search every combination on a grid, and then let
  **Bayesian optimisation** search more cleverly.
- *Does cleaning the image help?*  We build a **pipeline** of steps — blur, median filter,
  non-local means, unsharp masking, and blemish removal by opening and closing — that we can
  switch on and off, and we measure what each one is really worth.
- *How do we compare a dozen pipelines without fooling ourselves?*  We add a **validation set**.

The coding in this lab is light.  Most of the work is in the **🤔 Think** puzzles: questions
about what the numbers mean.  Write your answers in the ✏️ cells — short sentences are fine.

**Textbook:** [Chapter 4](https://obooks.tech/imageprocessing/chapter4.html) (the segmentation pipeline),
[Appendix B](https://obooks.tech/imageprocessing/appendixb.html) (Bayesian optimisation),
[Chapters 2 and 3](https://obooks.tech/imageprocessing/chapter2.html) (the filters and morphology), and
[Appendix C](https://obooks.tech/imageprocessing/appendixc.html) (validation and data leakage).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

Some cells take a minute or two to run; they say so.

## 0. Setup — Get the Data

As in Lab 2: copy the repository, load the images and masks, and convert to grayscale.  We also
install `scikit-optimize`, the Bayesian optimisation library.

In [ ]:
!pip install scikit-optimize --quiet
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)

gray_all = 0.299 * X[:, 0] + 0.587 * X[:, 1] + 0.114 * X[:, 2]        # (200, 256, 256)
print("Loaded", X.shape[0], "images")

### Three sets of images: train, validation, test

Lab 2 split the images into 160 training and 40 test images.  We keep **exactly the same 40
test images**, so our results can be compared with Lab 2's.  But this time we split the 160
again:

| Set | Images | Used for |
|---|---|---|
| **train** | 120 | tuning the thresholds |
| **validation** | 40 | comparing pipelines and choosing the best one |
| **test** | 40 | one final score, at the very end |

📖 Book: [Appendix C — Validation Set and Cross-Validation](https://obooks.tech/imageprocessing/appendixc.html#validation-set-and-cross-validation)

In [ ]:
from sklearn.model_selection import train_test_split

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))

# The same split as Lab 2 → the same 40 test images
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)

# Split the other 160 into 120 train + 40 validation
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)

print("train:", len(train_idx), "  validation:", len(val_idx), "  test:", len(test_idx))

### 🤔 Think 1 — Why a third set?

In Lab 2 we tuned on the training images and looked at the test images once.  In this lab we
are going to try **a dozen different pipelines** and keep the best one.

Suppose we skipped the validation set and picked the pipeline with the best **test** score.
Why would that test score then be too optimistic?

📖 Book: [Appendix C — Data Leakage](https://obooks.tech/imageprocessing/appendixc.html#data-leakage)

✏️ **Your answer:** *(double-click this cell and type here)*

## 1. Recap: the Segmenter and Its Score

These are Lab 2's functions, with two small changes:
- `image_dice` averages the cytoplasm and nucleus Dice of one image, as in Lab 2.
- `image_scores` returns the Dice of **every** image, so we can compare pipelines image by image.
- `mask_steps` is a list of clean-up functions applied to each prediction.  It is empty for
  now; we fill it in Section 4.

📖 Book: [§4.3 Stage 1 — Grayscale Thresholding](https://obooks.tech/imageprocessing/chapter4.html#sec-stage1)

In [ ]:
def segment(gray, t_nucleus, t_background):
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus] = 2
    pred[(gray >= t_nucleus) & (gray < t_background)] = 1
    return pred

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def image_scores(grays, masks, t_nucleus, t_background, mask_steps=[]):
    scores = []
    for gray, mask in zip(grays, masks):
        pred = segment(gray, t_nucleus, t_background)
        for step in mask_steps:
            pred = step(pred)
        scores.append(image_dice(pred, mask))
    return np.array(scores)

def mean_dice(grays, masks, t_nucleus, t_background, mask_steps=[]):
    return image_scores(grays, masks, t_nucleus, t_background, mask_steps).mean()

print(f"Lab 2 settings (0.40, 0.85), training Dice: {mean_dice(gray_all[train_idx], y[train_idx], 0.40, 0.85):.4f}")

## 2. Grid Search: Try Every Combination

The most straightforward way to tune two thresholds: pick 11 candidate values for each, try
all 11 × 11 = 121 pairs, and keep the best.  This is called **grid search**.

### 🧩 Puzzle 1 — Fill the grid

`scores[i, j]` should hold the training Dice for the `i`-th nucleus threshold and the `j`-th
background threshold.  *(Takes about 15 seconds.)*

📖 Book: [§4.4 Stage 2 — Optimising the Thresholds](https://obooks.tech/imageprocessing/chapter4.html#sec-stage2)

<details><summary><b>💡 Hint</b> (click to open)</summary>

Inside the loops, the two thresholds being tried are called `t_nuc` and `t_bg`.

</details>

In [ ]:
t_nuc_values = np.round(np.linspace(0.20, 0.60, 11), 2)   # 0.20, 0.24, ..., 0.60
t_bg_values  = np.round(np.linspace(0.80, 1.00, 11), 2)   # 0.80, 0.82, ..., 1.00

scores = np.zeros((11, 11))
start = time.time()
for i, t_nuc in enumerate(t_nuc_values):
    for j, t_bg in enumerate(t_bg_values):
        scores[i, j] = mean_dice(gray_all[train_idx], y[train_idx], ___, ___)
grid_seconds = time.time() - start
print(f"121 evaluations took {grid_seconds:.1f} seconds")

# ── Check ──
assert abs(scores[5, 2] - mean_dice(gray_all[train_idx], y[train_idx], t_nuc_values[5], t_bg_values[2])) < 1e-9, \
    "scores[i, j] should use t_nuc_values[i] and t_bg_values[j]"
print("✅ Puzzle 1 solved!")

### 🧩 Puzzle 2 — Find the best cell

`np.argmax` gives the position of the largest value, but for a 2-D array it counts cells as if
the array were one long row.  `np.unravel_index` turns that count back into a (row, column) pair.

📖 Book: [§1.2.4 Aggregation Functions](https://obooks.tech/imageprocessing/chapter1.html#aggregation-functions)

<details><summary><b>💡 Hint</b> (click to open)</summary>

You used the same function in Lab 2, Puzzle 10, to find the best threshold.

</details>

In [ ]:
i, j = np.unravel_index(np.___(scores), scores.shape)
best_nuc, best_bg = t_nuc_values[i], t_bg_values[j]
print(f"Best on the grid: t_nucleus = {best_nuc}, t_background = {best_bg}, training Dice = {scores[i, j]:.4f}")

plt.figure(figsize=(6, 5))
plt.imshow(scores, cmap='viridis', origin='lower', extent=[0.79, 1.01, 0.18, 0.62], aspect='auto')
plt.colorbar(label='mean Dice (training images)')
plt.plot(0.85, 0.40, 'wo', markersize=10)          # Lab 2's choice
plt.plot(best_bg, best_nuc, 'r*', markersize=15)  # grid search's choice
plt.xlabel('t_background')
plt.ylabel('t_nucleus')
plt.title('Training Dice for every threshold pair (○ Lab 2, ★ best)')
plt.show()

# ── Check ──
assert scores[i, j] == scores.max(), "Look for the position of the largest score"
print("✅ Puzzle 2 solved!")

### 🤔 Think 2 — The best value is on the edge

The best `t_background` is **1.00** — the very last column of the grid, and a jump of almost
0.05 in Dice over Lab 2's choice.

1. In words, what does `t_background = 1.00` mean?  Which pixels count as background?
2. Usually, when the best value sits on the edge of a grid, you should extend the grid.  Should
   we try `t_background = 1.05`?  (Run `print(gray_all.max())` in a new cell if you are unsure.)
3. Lab 1's histogram showed how bright cytoplasm can be.  Why was 0.85 such a costly choice?

📖 Book: [§4.3.2 The Intensity Histogram](https://obooks.tech/imageprocessing/chapter4.html#the-intensity-histogram)

✏️ **Your answer:** *(double-click this cell and type here)*

### 🧩 Puzzle 3 — What would a bigger grid cost?

Two knobs with 11 values each needed 121 evaluations.  Later in this lab our pipeline will have
**five** knobs.  With 11 values for each knob, how many evaluations would a grid need?

<details><summary><b>💡 Hint</b> (click to open)</summary>

Two knobs: 11 × 11 = 11 ** 2.  Five knobs: …

</details>

In [ ]:
n_evaluations = 11 ** ___
seconds_each  = grid_seconds / 121

print(f"Evaluations needed: {n_evaluations:,}")
print(f"Estimated time:     {n_evaluations * seconds_each / 3600:.1f} hours")

# ── Check ──
assert n_evaluations == 161051, "One factor of 11 per knob"
print("✅ Puzzle 3 solved!")

And that estimate is optimistic: the extra knobs (blur, opening, closing) make every single
evaluation slower too.  Grid search does not scale — every new knob **multiplies** the cost.

## 3. Bayesian Optimisation: Search Cleverly

**Bayesian optimisation** tries a few threshold pairs, fits a smooth guess of the whole Dice
surface through the results (a *Gaussian process*), and uses that guess to choose the next pair
to try — either where the guess is high, or where it is very unsure.  Each new result improves
the guess.

`gp_minimize` from `scikit-optimize` does all of this.  It needs:
- an **objective** function that takes a list of knob values and returns one number, and
- a **search space**: the range of each knob.

📖 Book: [Appendix B — The Bayesian Optimization Algorithm](https://obooks.tech/imageprocessing/appendixb.html#the-bayesian-optimization-algorithm) and
[Appendix B — Practical Implementation with Scikit-Optimize](https://obooks.tech/imageprocessing/appendixb.html#practical-implementation-with-scikit-optimize)

### 🧩 Puzzle 4 — The objective

`gp_minimize` always looks for the **smallest** value.  We want the **largest** Dice.

<details><summary><b>💡 Hint</b> (click to open)</summary>

Turning a maximum into a minimum takes one character.

</details>

In [ ]:
from skopt import gp_minimize

def objective(params):
    t_nucleus, t_background = params
    return ___mean_dice(gray_all[train_idx], y[train_idx], t_nucleus, t_background)

# ── Check ──
assert objective([0.40, 1.00]) < objective([0.40, 0.85]), "A better Dice should give a smaller objective"
print("✅ Puzzle 4 solved!")

Now run it with a budget of **30** evaluations — a quarter of the grid's 121.  *(About 10 seconds.)*

In [ ]:
result = gp_minimize(objective, [(0.20, 0.60), (0.80, 1.00)], n_calls=30, random_state=0)

bo_nuc, bo_bg = result.x
print(f"Bayesian optimisation: t_nucleus = {bo_nuc:.3f}, t_background = {bo_bg:.3f}, training Dice = {-result.fun:.4f}")
print(f"Grid search:           t_nucleus = {best_nuc:.3f}, t_background = {best_bg:.3f}, training Dice = {scores.max():.4f}")

tried = np.array(result.x_iters)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(scores, cmap='viridis', origin='lower', extent=[0.79, 1.01, 0.18, 0.62], aspect='auto')
axes[0].plot(tried[:, 1], tried[:, 0], 'w.', markersize=10)
axes[0].plot(bo_bg, bo_nuc, 'r*', markersize=15)
axes[0].set_xlabel('t_background')
axes[0].set_ylabel('t_nucleus')
axes[0].set_title('The 30 pairs Bayesian optimisation tried')

axes[1].plot(range(1, 31), -np.minimum.accumulate(result.func_vals), 'o-')
axes[1].axhline(scores.max(), color='red', linestyle='--')
axes[1].set_xlabel('evaluation number')
axes[1].set_ylabel('best training Dice so far')
axes[1].set_title('Convergence (red line = best on the grid)')
plt.tight_layout()
plt.show()

### 🤔 Think 3 — Grid search or Bayesian optimisation?

1. Did Bayesian optimisation find the same answer as the grid?  After how many evaluations
   did its convergence curve reach the red line?
2. Look at the white dots.  Where did it spend its evaluations, and where did it hardly look?
   Why is that a sensible way to spend a limited budget?
3. For **these two** thresholds the grid took only seconds.  Give one advantage the grid
   still has over Bayesian optimisation.  (Hint: what does the heatmap show you that 30 dots
   do not?)

📖 Book: [Appendix B — When to Use Each Approach](https://obooks.tech/imageprocessing/appendixb.html#when-to-use-each-approach)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. A Pipeline of Processing Steps

Can we beat plain thresholding by cleaning things up?  We have two kinds of steps:

**Image steps** — change the grayscale image *before* thresholding:

| Step | What it does | Book |
|---|---|---|
| `blur` | Gaussian blur, σ = 1: averages each pixel with its neighbours | [Gaussian Filtering](https://obooks.tech/imageprocessing/chapter2.html#gaussian-filtering-blur-based-denoising) |
| `median` | 3 × 3 median filter: removes isolated outlier pixels | [Median Filtering](https://obooks.tech/imageprocessing/chapter2.html#median-filtering) |
| `nlm` | Non-local means: averages pixels whose surroundings look alike | [Non-Local Means](https://obooks.tech/imageprocessing/chapter2.html#non-local-means-denoising) |
| `sharpen` | Unsharp masking: boosts edges and fine detail | [§2.7 Unsharp Masking](https://obooks.tech/imageprocessing/chapter2.html#unsharp-masking-enhancing-image-contrast-and-sharpness) |

**Mask steps** — tidy the predicted mask *after* thresholding (blemish removal):

| Step | What it does | Book |
|---|---|---|
| `open_nucleus` | Opening on the nucleus mask: removes small specks of "nucleus"; they become cytoplasm | [§3.2.3 Opening](https://obooks.tech/imageprocessing/chapter3.html#opening-erosion-followed-by-dilation) |
| `close_nucleus` | Closing on the nucleus mask: fills small gaps and notches inside a nucleus | [§3.2.4 Closing](https://obooks.tech/imageprocessing/chapter3.html#closing-dilation-followed-by-erosion) |

A pipeline is just two **lists** of these functions, applied in order.  To add a step, put it
in the list; to remove it, take it out.

In [ ]:
from scipy.ndimage import gaussian_filter, median_filter
from skimage.restoration import denoise_nl_means
from skimage.filters import unsharp_mask
from skimage import morphology

# ── Image steps: grayscale image in, grayscale image out ──
def blur(gray):
    return gaussian_filter(gray, sigma=1)

def median(gray):
    return median_filter(gray, size=3)

def nlm(gray):
    return denoise_nl_means(gray, h=0.05, patch_size=5, patch_distance=6)

def sharpen(gray):
    return unsharp_mask(gray, radius=2, amount=1.0)

# ── Mask steps: predicted mask in, predicted mask out ──
def open_nucleus(pred, radius=2):
    nucleus = (pred == 2)
    kept = morphology.opening(nucleus, morphology.disk(radius))
    out = pred.copy()
    out[nucleus & ~kept] = 1          # removed specks become cytoplasm
    return out

def close_nucleus(pred, radius=2):
    nucleus = (pred == 2)
    filled = morphology.closing(nucleus, morphology.disk(radius))
    out = pred.copy()
    out[filled & (pred == 1)] = 2     # filled gaps become nucleus
    return out

Here is what each step does to image 7:

In [ ]:
gray7 = gray_all[7]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, step in zip(axes, [None, blur, median, nlm, sharpen]):
    ax.imshow(gray7 if step is None else step(gray7), cmap='gray', vmin=0, vmax=1)
    ax.set_title('original' if step is None else step.__name__)
    ax.axis('off')
plt.show()

pred7 = segment(gray7, 0.40, 1.00)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (title, mask) in zip(axes, [('true mask', y[7]), ('threshold only', pred7),
                                     ('+ open_nucleus', open_nucleus(pred7, 4)),
                                     ('+ close_nucleus', close_nucleus(pred7, 4))]):
    ax.imshow(mask, cmap='viridis', vmin=0, vmax=2)
    ax.set_title(title)
    ax.axis('off')
plt.show()

Look closely at the masks (radius 4 here): opening clears many of the yellow specks from the
cytoplasm, but closing also **joins** nearby specks into larger false nucleus.  Whether a step
helps on average is a question only the numbers can answer — that is the next part.

### Testing a pipeline fairly

`evaluate` runs one pipeline from start to finish:

1. apply the image steps to all 200 images,
2. **re-tune both thresholds** on the training images with Bayesian optimisation
   (25 evaluations), using the mask steps,
3. report training and validation Dice, and return the validation Dice of every image.

In [ ]:
def preprocess(grays, image_steps):
    out = []
    for gray in grays:
        for step in image_steps:
            gray = step(gray)
        out.append(gray)
    return np.array(out)

def evaluate(name, image_steps, mask_steps):
    images = preprocess(gray_all, image_steps)

    def objective(params):
        return -mean_dice(images[train_idx], y[train_idx], params[0], params[1], mask_steps)

    result = gp_minimize(objective, [(0.20, 0.60), (0.80, 1.00)], n_calls=25, random_state=0)
    t_nucleus, t_background = result.x
    val = image_scores(images[val_idx], y[val_idx], t_nucleus, t_background, mask_steps)
    print(f"{name:24s} thresholds ({t_nucleus:.2f}, {t_background:.2f})   train {-result.fun:.4f}   validation {val.mean():.4f}")
    return val

### 🧩 Puzzle 5 — Build your own pipeline

Make a pipeline that **removes outlier pixels with the median filter, then sharpens**, and
afterwards **removes nucleus specks with opening**.  *(About 10 seconds.)*

<details><summary><b>💡 Hint</b> (click to open)</summary>

Put the function names in the lists, in the order they should run — no brackets after the names,
because we are listing the functions, not calling them.

</details>

In [ ]:
my_image_steps = [___, ___]
my_mask_steps  = [___]

val_mine = evaluate('my pipeline', my_image_steps, my_mask_steps)

# ── Check ──
assert my_image_steps == [median, sharpen], "Median filter first, then sharpen"
assert my_mask_steps == [open_nucleus], "Only opening after thresholding"
print("✅ Puzzle 5 solved!")

### The ablation study

Now we test many pipelines, adding or removing one step at a time — an **ablation study**.
*(About 2 minutes; non-local means is the slow one.)*  Feel free to add your own lines to `pipelines`.

In [ ]:
pipelines = {
    'none':                   ([],                []),
    'blur':                   ([blur],            []),
    'median':                 ([median],          []),
    'nlm':                    ([nlm],             []),
    'sharpen':                ([sharpen],         []),
    'blur, sharpen':          ([blur, sharpen],   []),
    'sharpen, blur':          ([sharpen, blur],   []),
    'open':                   ([],                [open_nucleus]),
    'close':                  ([],                [close_nucleus]),
    'open, close':            ([],                [open_nucleus, close_nucleus]),
    'median + open, close':   ([median],          [open_nucleus, close_nucleus]),
}

val_scores = {}
for name, (image_steps, mask_steps) in pipelines.items():
    val_scores[name] = evaluate(name, image_steps, mask_steps)

### 🤔 Think 4 — Why re-tune the thresholds every time?

`evaluate` runs a fresh Bayesian optimisation for every pipeline.  It would be much faster to
keep the thresholds from Section 3 (about 0.40 and 1.00) for all of them.

Look at the thresholds chosen for `nlm` and for `sharpen, blur`.  Why would re-using one
fixed pair of thresholds be an **unfair** test of a step?

✏️ **Your answer:** *(double-click this cell and type here)*

### 🤔 Think 5 — Sharpening made things worse

`sharpen` scores clearly **below** `none`, even though a sharpened image looks crisper to us.

1. Unsharp masking adds back an amplified copy of the fine detail.  What does that do to the
   grain *inside* the cytoplasm and to the pixels right beside an edge?  How does a threshold
   react?
2. `blur, sharpen` and `sharpen, blur` use the same two steps.  Do they score the same?  Why
   does the order matter?

📖 Book: [§2.7.1 What Unsharp Masking Does](https://obooks.tech/imageprocessing/chapter2.html#what-unsharp-masking-does)

✏️ **Your answer:** *(double-click this cell and type here)*

### 🧩 Puzzle 6 — Is the difference real?

The best pipelines beat `none` by only a few thousandths on the validation set.  Since
`val_scores` keeps the Dice of every validation image, we can ask a sharper question:
**on how many of the 40 images** did `median + open, close` beat `none`?

<details><summary><b>💡 Hint</b> (click to open)</summary>

The comparison gives an array of `True`/`False`.  Count the `True`s the same way you counted
pixels in Lab 2.

</details>

In [ ]:
better = (val_scores['median + open, close'] > val_scores['none']).___()
worse  = (val_scores['median + open, close'] < val_scores['none']).___()

print(f"Mean validation Dice:  none {val_scores['none'].mean():.4f}   median + open, close {val_scores['median + open, close'].mean():.4f}")
print(f"Better on {better} of 40 images, worse on {worse}, the same on {40 - better - worse}")

plt.scatter(val_scores['none'], val_scores['median + open, close'], alpha=0.7)
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("Dice, no processing")
plt.ylabel("Dice, median + open, close")
plt.title("Each dot is one validation image")
plt.show()

# ── Check ──
assert better + worse <= 40, "Count the images, not the pixels"
assert better == np.sum(val_scores['median + open, close'] > val_scores['none']), "Count the True values"
print("✅ Puzzle 6 solved!")

### 🤔 Think 6 — Small gains, small sets

1. `median + open, close` beats `none` by less than 0.004 on average — but on how many of the
   40 images does it win?  Is a gain that small but that *consistent* more believable than the
   average alone suggests?  Would you bet that the whole table keeps its order on 40 *new*
   images?
2. Compare the training and validation columns of the table.  Does the pipeline that is best on
   training also win on validation?
3. The validation Dice is **higher** than the training Dice for almost every pipeline.  Does
   that mean the pipelines work better on new images?  What else could explain it?

📖 Book: [Appendix C — Cross-Validation](https://obooks.tech/imageprocessing/appendixc.html#sec-cross-validation)

✏️ **Your answer:** *(double-click this cell and type here)*

## 5. Five Knobs at Once

In Section 4 every step had **fixed** settings (σ = 1, radius = 2).  Maybe opening helps a lot
with a bigger radius, or blur helps with a smaller σ.  Let's give the pipeline five knobs and
let Bayesian optimisation turn all of them together:

| Knob | Range | 0 means |
|---|---|---|
| `t_nucleus` | 0.20 – 0.60 | |
| `t_background` | 0.80 – 1.00 | |
| `sigma` (blur) | 0 – 3 | no blur |
| `open_radius` | 0 – 5 | no opening |
| `close_radius` | 0 – 5 | no closing |

Puzzle 3 showed that a grid over five knobs would take hours.  Bayesian optimisation gets
**40** evaluations.

📖 Book: [§4.5 Stage 3 — Cleaning the Image](https://obooks.tech/imageprocessing/chapter4.html#sec-stage3) and
[§4.6 Stage 4 — Targeted Cleanup](https://obooks.tech/imageprocessing/chapter4.html#sec-stage4)

### 🧩 Puzzle 7 — The search space

Radii are whole numbers, so they use `Integer` instead of `Real`.  Fill in the missing knob.

In [ ]:
from skopt.space import Real, Integer

def predict(gray, t_nucleus, t_background, sigma, open_radius, close_radius):
    if sigma > 0:
        gray = gaussian_filter(gray, sigma)
    pred = segment(gray, t_nucleus, t_background)
    if open_radius > 0:
        pred = open_nucleus(pred, open_radius)
    if close_radius > 0:
        pred = close_nucleus(pred, close_radius)
    return pred

def pipeline_dice(idx, params):
    return np.mean([image_dice(predict(gray_all[i], *params), y[i]) for i in idx])

space = [Real(0.20, 0.60),     # t_nucleus
         Real(0.80, 1.00),     # t_background
         Real(0, 3),           # sigma
         Integer(0, 5),        # open_radius
         ___]      # close_radius

# ── Check ──
assert len(space) == 5 and space[4].low == 0 and space[4].high == 5, "close_radius: whole numbers from 0 to 5"
print("✅ Puzzle 7 solved!")

Run the search.  *(About 2–3 minutes.)*

In [ ]:
result5 = gp_minimize(lambda params: -pipeline_dice(train_idx, params), space, n_calls=40, random_state=0)

best5 = result5.x
print("Best settings found:")
for name, value in zip(['t_nucleus', 't_background', 'sigma', 'open_radius', 'close_radius'], best5):
    print(f"  {name:13s} = {value:.3f}")
print(f"Training Dice:   {-result5.fun:.4f}")
print(f"Validation Dice: {pipeline_dice(val_idx, best5):.4f}")

### 🤔 Think 7 — What did the optimiser decide?

1. Look at `sigma`.  What did the optimiser decide about blurring?
2. In the ablation study, `open, close` with radius 2 barely beat `none`.  Which radii did the
   optimiser pick, and how does its validation Dice compare with the ablation table?  What does
   that tell you about concluding "this step is useless" after trying one setting?
3. One radius landed on the edge of its range (5).  What would you try next?
4. Five knobs fitted to 120 images is more freedom than two.  Why is it important that we
   judge the result on the validation images, not the training images?

✏️ **Your answer:** *(double-click this cell and type here)*

## 6. The Final Test — Once

We choose our final segmenter using the **validation** scores: the five-knob pipeline.  Now,
and only now, we score it on the 40 test images — the same 40 that Lab 2 used — together with
Lab 2's settings and the two tuned thresholds alone.  We also check the N/C ratio.

In [ ]:
def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

candidates = {
    'Lab 2 (0.40, 0.85, blur σ=2)':  [0.40, 0.85, 2, 0, 0],
    'two tuned thresholds':          [bo_nuc, bo_bg, 0, 0, 0],
    'five-knob pipeline':            best5,
}

print(f"{'':32s} test Dice   N/C R²")
for name, params in candidates.items():
    preds   = [predict(gray_all[i], *params) for i in test_idx]
    dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
    nc_pred = np.array([nc_ratio(p) for p in preds])
    r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
    print(f"{name:32s} {dice:.4f}    {r2:7.3f}")

### 🤔 Think 8 — Reading the final table

1. The five-knob pipeline scored about 0.81 on validation.  Is its test Dice higher or lower?
   Give two reasons why a validation score tends to be optimistic once you have used it to
   choose between many options.
2. Rank the three rows by test Dice.  Which single change since Lab 2 was worth the most?
3. Compare the N/C R² of the last two rows.  Did the better Dice give a better N/C ratio?  Is
   the N/C ratio — the number a pathologist needs — usable yet?  What is still holding
   thresholding back?

📖 Book: [§4.7 Comparing All Four Stages](https://obooks.tech/imageprocessing/chapter4.html#comparing-all-four-stages)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **Grid search** tries every combination.  It maps the whole surface and is the right tool
  for one or two cheap knobs, but its cost multiplies with every knob.
- **Bayesian optimisation** spends a small budget where the results look promising.  It found
  the grid's answer in a quarter of the evaluations, and it tuned five knobs where a grid would
  take hours.
- **A pipeline is a list of steps.**  Ablation — adding or removing one step at a time — shows
  what each one is worth.  Re-tune the thresholds for every pipeline, or the comparison is unfair.
- **Not every "improvement" improves.**  Sharpening looks better and scores worse.  A step tried
  at one setting can look useless when it only had the wrong setting.
- **Train, validation, test.**  Tune on train, choose on validation, report test once.  Every
  choice made by looking at a set makes that set's score optimistic.

**The ceiling, again:** the best pipeline improved Dice from Lab 2's 0.72 to about 0.77, but
the N/C ratio is still unreliable.  Thresholding looks at one brightness value per pixel.  In
the next labs we give each pixel more to go on — colour, texture and its neighbourhood — and
let a classifier learn the rule.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run Section 5 |
|---|---|
| A | `n_calls=15` instead of 40 |
| B | `random_state=1` instead of 0 |
| C | Allow radii up to 8: `Integer(0, 8)` |

1. Did the best settings change?  Did the validation Dice change?
2. If two runs with different random seeds find different "best" settings with almost the same
   score, what does that tell you about the Dice surface — and about how precisely the knobs matter?
3. Which result in this lab surprised you most, and why?